# Safe Drive cleanup and reversible seed reset

This non-canonical utility inventories one exact model-family/seed package and, only after an exact typed confirmation, moves it to a timestamped archive with a restore ledger. It never deletes artifacts. The default is a zero-mutation dry run. Shared data, global caches, and unrelated seeds are never selected.

If OOD or RQ1 evidence depends on the seed, the dry run stops unless `INCLUDE_DOWNSTREAM=True` is chosen explicitly. Inspect and back up the displayed plan before applying it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

MODEL_FAMILY = 'gemma3'  # gemma3 or qwen2_5_vl
SEED = 42  # 42, 43, or 44
INCLUDE_DOWNSTREAM = False
DRIVE_ROOTS = {
    'gemma3': Path('/content/drive/MyDrive/em-displacement-vlm'),
    'qwen2_5_vl': Path('/content/drive/MyDrive/em-displacement-vlm-qwen2-5-vl-3b'),
}
DRIVE_PROJECT = DRIVE_ROOTS[MODEL_FAMILY]
ARCHIVE_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
if not DRIVE_PROJECT.is_dir():
    raise SystemExit(f'Exact Drive project root is missing: {DRIVE_PROJECT}')
print('Scope:', MODEL_FAMILY, 'seed', SEED)
print('Drive project:', DRIVE_PROJECT)
print('Archive timestamp:', ARCHIVE_TIMESTAMP)

## Resolve a clean immutable code checkout

Runtime clones under `/content` are disposable and are not Drive evidence. This cell refuses a dirty clone or an unexpected Git remote; it never pulls, resets, cleans, or repairs a checkout in place.

In [ ]:
import subprocess

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
REPO_REF = 'main'  # Prefer an exact 40-character commit for a recorded archive.
if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise SystemExit(f'{REPO_DIR} is not the canonical Git clone; restart Colab.')
    origin = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip()
    if origin.rstrip('/') not in {REPO_URL.rstrip('/'), REPO_URL.removesuffix('.git')}:
        raise SystemExit(f'Unexpected origin {origin!r}; restart Colab.')
    dirty = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'status', '--porcelain=v1', '--untracked-files=all'],
        text=True,
    ).strip()
    if dirty:
        raise SystemExit('Runtime clone is dirty; restart Colab instead of repairing it.')
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', '--tags', 'origin'])
else:
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)])
target = 'origin/main' if REPO_REF == 'main' else REPO_REF
commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{target}^{{commit}}'], text=True
).strip()
subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', commit])
%cd {REPO_DIR}
print('Resolved commit:', commit)

In [ ]:
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[dev]'])

## Inventory and dry run

The plan contains exact relative paths, sizes, and content hashes. Keep `INCLUDE_DOWNSTREAM=False` unless you intentionally want to invalidate and archive dependent OOD/RQ1 evidence too.

In [ ]:
import json
from em_displacement_vlm.maintenance import build_archive_plan

PLAN = build_archive_plan(
    DRIVE_PROJECT,
    model_family=MODEL_FAMILY,
    seed=SEED,
    include_downstream=INCLUDE_DOWNSTREAM,
    timestamp=ARCHIVE_TIMESTAMP,
)
print(json.dumps(PLAN.to_dict(), indent=2, sort_keys=True))
print('\nDRY_RUN: no Drive file was moved or deleted.')
print('Required confirmation:', PLAN.confirmation)

## Optional reversible archive

Leave `APPLY_ARCHIVE=False` for inventory-only use. To apply exactly the displayed plan, set it to true and paste the complete confirmation printed above. If any source changed after the dry run, application stops and asks for a new plan.

In [ ]:
from em_displacement_vlm.maintenance import apply_archive_plan

APPLY_ARCHIVE = False
CONFIRMATION = ''
if not APPLY_ARCHIVE:
    print('Dry run only. No Drive file was moved or deleted.')
else:
    ledger = apply_archive_plan(PLAN, confirmation=CONFIRMATION)
    print('Archive complete. Nothing was deleted.')
    print('Restore ledger:', ledger)